# 1. Returns to education in a cross-section (WAGE1)

A worked example of the skill's 7-step workflow on Wooldridge's WAGE1 data (526 workers, 1976 CPS).
**Question:** how much does one more year of education raise hourly wages, and how much should we
trust that number as a causal effect?

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo

from _setup import *  # noqa: F401,F403  (adds the skill's scripts to the path)
import data_prep as dp
import diagnostics as dg
import reporting as rp
import specification as sp

## Step 1 — Frame the question and expected signs
- y = log hourly wage, so coefficients are approximate percentage effects (log-lin).
- educ: positive (human capital). exper: positive but concave (add exper²). tenure: positive.
- Goal: *estimate an effect*. The threat is omitted **ability**, which raises both schooling and
  wages, so OLS is probably biased **upward**. We will quantify how fragile the estimate is.

## Step 2 — Data structure: cross-section, continuous y → `ols-regression.md`

## Step 3 — Audit the data

In [2]:
df = woo.data("wage1")
cols = ["wage", "educ", "exper", "tenure", "female", "married", "nonwhite"]
dp.audit(df[cols], show=False).head(8)

,dtype,missing_%,n_unique,examples
column,,,,
wage,float64,0.0,241,3.0999999046325684 | 3.240000009536743 | 3.0
educ,int64,0.0,18,11 | 12 | 8
exper,int64,0.0,51,2 | 22 | 44
tenure,int64,0.0,34,0 | 2 | 28
female,int64,0.0,2,1 | 0
married,int64,0.0,2,0 | 1
nonwhite,int64,0.0,2,0 | 1


In [3]:
miss = dp.missing_report(df, cols, show=True)


Missing data: 526 of 526 rows complete on 7 columns (100.0%)
none
Most frequent patterns (1 = missing):
 wage  educ  exper  tenure  female  married  nonwhite  rows
    0     0      0       0       0        0         0   526


## Step 4 — Specify and estimate (simplest model first, robust errors by default)

In [4]:
m1 = smf.ols("np.log(wage) ~ educ", df).fit(cov_type="HC3")
m2 = smf.ols("np.log(wage) ~ educ + exper + I(exper**2) + tenure", df).fit(cov_type="HC3")
m3 = smf.ols("np.log(wage) ~ educ + exper + I(exper**2) + tenure + female + married + nonwhite",
             df).fit(cov_type="HC3")

## Step 5 — Diagnose (cross-section: no autocorrelation tests, row order is arbitrary)

In [5]:
diag = dg.ols_diagnostics(m3)


OLS diagnostics (N=526, R²=0.436, adj. R²=0.429, condition number=4498.6)
                    Test                      H0  Statistic  p-value                                                                 Conclusion
   Ramsey RESET (ŷ², ŷ³) correct functional form     6.9778   0.0010           Likely misspecification: respecify BEFORE fixing standard errors
           Breusch-Pagan        homoskedasticity    16.9647   0.0176                                  Heteroskedasticity: use HC3 (or WLS/FGLS)
            White (full)        homoskedasticity    48.9570   0.0213                                                         Heteroskedasticity
White simplified (ŷ, ŷ²)        homoskedasticity     5.0823   0.0788                                                                No evidence
             Jarque-Bera        normal residuals    25.9956   0.0000 Non-normal (skewness -0.07, kurtosis 4.08); of little concern with large N
Cross-section: autocorrelation tests skipped (row order is ar

**RESET rejects** (p ≈ 0.001): the functional form is wrong, and the workflow says to respecify
*before* worrying about standard errors. Theory suggests two candidates: the marriage premium
differs by gender (an interaction), and returns to schooling may rise with the level of
schooling (credentials, college premium), so add educ² and tenure².

In [6]:
m4 = smf.ols("np.log(wage) ~ educ + I(educ**2) + exper + I(exper**2) + tenure + I(tenure**2)"
             " + female*married + nonwhite", df).fit(cov_type="HC3")
diag4 = dg.ols_diagnostics(m4)


OLS diagnostics (N=526, R²=0.476, adj. R²=0.465, condition number=8775.8)
                    Test                      H0  Statistic  p-value                                                                 Conclusion
   Ramsey RESET (ŷ², ŷ³) correct functional form     0.8809   0.4150                                            No evidence of misspecification
           Breusch-Pagan        homoskedasticity    14.3569   0.1573                                                                No evidence
            White (full)        homoskedasticity    65.9978   0.1472                                                                No evidence
White simplified (ŷ, ŷ²)        homoskedasticity     6.0865   0.0477                                                         Heteroskedasticity
             Jarque-Bera        normal residuals    37.9987   0.0000 Non-normal (skewness -0.07, kurtosis 4.31); of little concern with large N
Cross-section: autocorrelation tests skipped (row order is ar

RESET no longer rejects — and neither do Breusch-Pagan nor the full White test, which both rejected
before. The "heteroskedasticity" was largely a symptom of the wrong functional form, which is why
the workflow checks the form first. Only the simplified White test is borderline (p ≈ 0.05); the
HC3 errors keep inference valid either way. With educ² in the model the return to schooling depends on the level of schooling:
∂log(wage)/∂educ = β₁ + 2β₂·educ. Evaluate it where it matters, with delta-method SEs.

In [7]:
for e in (10, 12, 16):
    t = m4.t_test(f"educ + {2*e}*I(educ ** 2) = 0")
    b, se = float(np.ravel(t.effect)[0]), float(np.ravel(t.sd)[0])
    print(f"Return at {e} years of schooling: {100*(np.exp(b)-1):.1f}% per extra year "
          f"(95% CI {100*(np.exp(b-1.96*se)-1):.1f}% to {100*(np.exp(b+1.96*se)-1):.1f}%)")

Return at 10 years of schooling: 6.4% per extra year (95% CI 5.1% to 7.9%)
Return at 12 years of schooling: 8.4% per extra year (95% CI 6.9% to 9.9%)
Return at 16 years of schooling: 12.4% per extra year (95% CI 9.6% to 15.1%)


## Step 6 — Robustness: specifications side by side

In [8]:
labels = {"educ": "Years of education", "I(educ ** 2)": "Education²", "female": "Female",
          "married": "Married", "female:married": "Female × married"}
rp.regression_table({"(1)": m1, "(2)": m2, "(3)": m3, "(4)": m4},
                    keep=["educ", "I(educ ** 2)", "female", "married", "female:married"], labels=labels)

,(1),(2),(3),(4)
,,,,
Years of education,0.083***,0.085***,0.080***,-0.028
,(0.008),(0.008),(0.008),(0.023)
Education²,,,,0.005***
,,,,(0.001)
Female,,,-0.291***,-0.099*
,,,(0.037),(0.058)
Married,,,0.055,0.206***
,,,(0.043),(0.058)
Female × married,,,,-0.295***


In the linear-in-education columns the average return is stable at about 8% across specifications.
Stability under observed controls is informative but not proof: what about **unobserved** ability?

### Sensitivity to an omitted confounder
These tools need a single coefficient, so they use the linear-in-education version of the final
specification, whose coefficient approximates the average return.
- **Oster (2019):** how strongly would unobservables have to drive selection, relative to the
  observed controls, to make the effect zero?
- **Cinelli-Hazlett (2020):** how much residual variance of both education and wages would a
  confounder need to explain?

In [9]:
m4_lin = smf.ols("np.log(wage) ~ educ + exper + I(exper**2) + tenure + I(tenure**2)"
                 " + female*married + nonwhite", df).fit()
osr = sp.oster(smf.ols("np.log(wage) ~ educ", df).fit(), m4_lin, "educ", show=True)
sens = sp.sensitivity(m4_lin, "educ", show=True)


Oster (R_max = 0.600, δ = 1.0): β̃ = 0.0785 → β* = 0.0764; identified set [0.0764, 0.0785]
δ needed for β = 0: 36.92 (> 1: robust by Oster's criterion)

Sensitivity of educ (Cinelli-Hazlett): partial R² = 20.913%, RV(q=1.0) = 39.87%, RV(q=1.0, α=0.05) = 34.59%
A confounder explaining 39.9% of the residual variance of both educ and the outcome would remove the estimate; 34.6% would make it insignificant.
Observed covariates (partial R² with outcome), for comparison:
exper            4.85%
I(exper ** 2)    4.39%
tenure           3.50%
female:married   3.34%
married          2.77%
I(tenure ** 2)   1.05%
female           0.74%
nonwhite         0.10%


## Step 7 — Interpret and report

Prediction for a specific worker (12 years of schooling, 10 of experience, 2 of tenure, male,
married, white). The **prediction interval** is for one person; the CI is for the average such
worker. In levels, report the smearing-corrected mean, not exp(prediction).

In [10]:
new = pd.DataFrame({"educ": [12], "exper": [10], "tenure": [2], "female": [0], "married": [1], "nonwhite": [0]})
sp.prediction_intervals(m4.model.fit(), new, log_y=True).round(2)

,prediction,se_mean,ci_lower,ci_upper,pi_lower,pi_upper,median_y,mean_y,pi_lower_y,pi_upper_y
0,1.72,0.04,1.64,1.79,0.95,2.48,5.57,5.99,2.58,11.98


**Report.** The simple log-wage equation fails RESET; allowing returns to rise with schooling and
the marriage premium to differ by gender fixes it. Returns to schooling increase with its level —
compare the estimates at 10, 12 and 16 years above — and average about 8% per year. That average
is stable across specifications; by Oster's criterion unobservables would need to be dozens of
times as important as the observed controls to erase it, and a confounder would need to explain
far more residual variance than any observed covariate does. Ability remains the obvious omitted
variable and probably biases the estimate upward; example 2 addresses it with instruments. For
one worker, the wage prediction interval is wide: individual wages vary far more than averages.